In [3]:
import yfinance as yf
print(yf.__version__)

StatementMeta(, ba18149d-9489-494d-b534-fe774a2cd2d3, 7, Finished, Available, Finished, False)

1.8.0


In [4]:
import io, requests, pandas as pd, yfinance as yf
from datetime import datetime, timezone

TICKERS = ["SPY", "EFA", "EEM", "AGG", "GLD", "CHSPI.SW", "IWDA.AS"]
START = "2019-01-01"
run_ts = datetime.now(timezone.utc).isoformat()

# 1. ETF prices (one row per ticker per day)
raw = yf.download(TICKERS, start=START, auto_adjust=False, group_by="ticker", progress=False)
frames = []
for t in TICKERS:
    df = raw[t].reset_index()
    df["ticker"] = t
    frames.append(df)
prices = pd.concat(frames, ignore_index=True)
prices.columns = [c.lower().replace(" ", "_") for c in prices.columns]
prices["_ingested_at"] = run_ts
spark.createDataFrame(prices).write.mode("overwrite").saveAsTable("bronze.prices_raw")

# 2. ECB reference rates: units of CHF / USD / GBP per 1 EUR
url = ("https://data-api.ecb.europa.eu/service/data/EXR/"
       f"D.CHF+USD+GBP.EUR.SP00.A?format=csvdata&startPeriod={START}")
fx = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))
fx = fx[["TIME_PERIOD", "CURRENCY", "OBS_VALUE"]]
fx.columns = ["date", "currency", "eur_rate"]
fx["_ingested_at"] = run_ts
spark.createDataFrame(fx).write.mode("overwrite").saveAsTable("bronze.fx_raw")

StatementMeta(, ba18149d-9489-494d-b534-fe774a2cd2d3, 8, Finished, Available, Finished, False)